# 第8章: ニューラルネット

第7章で取り組んだポジネガ分類を題材として、ニューラルネットワークで分類モデルを実装する。なお、この章ではPyTorchやTensorFlow、JAXなどの深層学習フレームワークを活用せよ。

## 70. 単語埋め込みの読み込み

事前学習済み単語埋め込みを活用し、$|V| \times d_\rm{emb}$ の単語埋め込み行列$\pmb{E}$を作成せよ。ここで、$|V|$は単語埋め込みの語彙数、$d_\rm{emb}$は単語埋め込みの次元数である。ただし、単語埋め込み行列の先頭の行ベクトル$\pmb{E}_{0,:}$は、将来的にパディング（`<PAD>`）トークンの埋め込みベクトルとして用いたいので、ゼロベクトルとして予約せよ。ゆえに、$\pmb{E}$の2行目以降に事前学習済み単語埋め込みを読み込むことになる。

もし、Google Newsデータセットの[学習済み単語ベクトル](https://drive.google.com/file/d/0B7XkCwpI5KDYNlNUTTlSS21pQmM/edit?usp=sharing)（300万単語・フレーズ、300次元）を全て読み込んだ場合、$|V|=3000001, d_\rm{emb}=300$になるはずである（ただ、300万単語の中には、殆ど用いられない稀な単語も含まれるので、語彙を削減した方がメモリの節約になる）。

また、単語埋め込み行列の構築と同時に、単語埋め込み行列の各行のインデックス番号（トークンID）と、単語（トークン）への双方向の対応付けを保持せよ。

In [3]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 73.4 MB/s eta 0:00:00


In [4]:
from os import major
import numpy as np
import gensim.downloader as api

# 学習済み単語ベクトルの読み込み
wv = api.load('word2vec-google-news-300')

# 使用する語彙数
max_vocab = 50000

# 語彙数と埋め込み次元
vocab_size = len(wv.index_to_key[:max_vocab]) + 1   # +1 は <PAD>
d_emb = wv.vector_size

# 埋め込み行列
E = np.zeros((vocab_size, d_emb), dtype=np.float32)

# token <-> id
token_to_id = {'<PAD>': 0}
id_to_token = {0: '<PAD>'}

# ベクトル格納
for i, token in enumerate(wv.index_to_key[:max_vocab], start=1):
    E[i] = wv[token]
    token_to_id[token] = i
    id_to_token[i] = token

print(E.shape)
print(token_to_id['Apple'])
print(id_to_token[1])

[==================================================] 100.0% 1662.8/1662.8MB downloaded
(50001, 300)
1815
</s>


## 71. データセットの読み込み

[General Language Understanding Evaluation (GLUE)](https://gluebenchmark.com/) ベンチマークで配布されている[Stanford Sentiment Treebank (SST)](https://dl.fbaipublicfiles.com/glue/data/SST-2.zip) をダウンロードし、訓練セット（train.tsv）と開発セット（dev.tsv）のテキストと極性ラベルと読み込み、全てのテキストをトークンID列に変換せよ。このとき、単語埋め込みの語彙でカバーされていない単語は無視し、トークン列に含めないことにせよ。また、テキストの全トークンが単語埋め込みの語彙に含まれておらず、空のトークン列となってしまう事例は、訓練セットおよび開発セットから削除せよ（このため、第7章の実験で得られた正解率と比較できなくなることに注意せよ）。

事例の表現方法は任意でよいが、例えば"contains no wit , only labored gags"がネガティブに分類される事例は、次のような辞書オブジェクトで表現すればよい。

```
{'text': 'contains no wit , only labored gags',
 'label': tensor([0.]),
 'input_ids': tensor([ 3475,    87, 15888,    90, 27695, 42637])}
```

この例では、`text`はテキスト、`label`は分類ラベル（ポジティブなら`tensor([1.])`、ネガティブなら`tensor([0.])`）、`input_ids`はテキストのトークン列をID列で表現している。

In [5]:
!wget https://dl.fbaipublicfiles.com/glue/data/SST-2.zip
!unzip SST-2.zip

--2026-05-28 06:00:21--  https://dl.fbaipublicfiles.com/glue/data/SST-2.zip
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 18.239.50.9, 18.239.50.18, 18.239.50.120, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|18.239.50.9|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7439277 (7.1M) [application/zip]
Saving to: ‘SST-2.zip’

SST-2.zip           100%[===================>]   7.09M  6.29MB/s    in 1.1s    

2026-05-28 06:00:23 (6.29 MB/s) - ‘SST-2.zip’ saved [7439277/7439277]

Archive:  SST-2.zip
   creating: SST-2/
  inflating: SST-2/dev.tsv           
   creating: SST-2/original/
  inflating: SST-2/original/README.txt  
  inflating: SST-2/original/SOStr.txt  
  inflating: SST-2/original/STree.txt  
  inflating: SST-2/original/datasetSentences.txt  
  inflating: SST-2/original/datasetSplit.txt  
  inflating: SST-2/original/dictionary.txt  
  inflating: SST-2/original/original_rt_snippets.txt  
  inflating: SST-2/original/sentim

In [6]:
import pandas as pd
import torch

# SST-2 の読み込み
train_df = pd.read_csv("SST-2/train.tsv", sep="\t")
dev_df = pd.read_csv("SST-2/dev.tsv", sep="\t")

# tokenize
def tokenize(text):
    return text.split()

# text -> token id列
def encode(text):

    ids = []

    for token in tokenize(text):

        # Word2Vec語彙に存在する単語のみ使用
        if token in token_to_id:
            ids.append(token_to_id[token])

    return torch.tensor(ids, dtype=torch.long)

# DataFrame -> dataset
def build_dataset(df):

    dataset = []

    for _, row in df.iterrows():

        text = row["sentence"]
        label = float(row["label"])

        # token id列へ変換
        input_ids = encode(text)

        # 空系列は除外
        if len(input_ids) == 0:
            continue

        example = {
            "text": text,
            "label": torch.tensor([label], dtype=torch.float),
            "input_ids": input_ids
        }

        dataset.append(example)

    return dataset

# dataset 作成
train_dataset = build_dataset(train_df)
dev_dataset = build_dataset(dev_df)

# 確認
print("train size:", len(train_dataset))
print("dev size:", len(dev_dataset))

print(train_dataset[0])

train size: 65018
dev size: 872
{'text': 'hide new secretions from the parental units ', 'label': tensor([0.]), 'input_ids': tensor([ 5785,    66,    18,    12, 15095,  1594])}


## 72. Bag of wordsモデルの構築

単語埋め込みの平均ベクトルでテキストの特徴ベクトルを表現し、重みベクトルとの内積でポジティブ及びネガティブを分類するニューラルネットワーク（ロジスティック回帰モデル）を設計せよ。

In [9]:
import torch
import torch.nn as nn

class BoWClassifier(nn.Module):

    def __init__(self, embedding_matrix):
        super().__init__()

        vocab_size, emb_dim = embedding_matrix.shape

        # 単語埋め込み層
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=emb_dim,
            padding_idx=0
        )

        # 学習済みベクトルをセット
        self.embedding.weight.data.copy_(
            torch.tensor(embedding_matrix)
        )

        # ロジスティック回帰
        self.linear = nn.Linear(emb_dim, 1)

    def forward(self, input_ids):

        # input_ids: [seq_len]

        # 単語埋め込み取得
        x = self.embedding(input_ids)
        # x.shape = [seq_len, emb_dim]

        # 平均ベクトル
        x = x.mean(dim=0)
        # x.shape = [emb_dim]

        # 線形変換
        logits = self.linear(x)
        # logits.shape = [1]

        return logits

## 73. モデルの学習

問題72で設計したモデルの重みベクトルを訓練セット上で学習せよ。ただし、学習中は単語埋め込み行列の値を固定せよ（単語埋め込み行列のファインチューニングは行わない）。また、学習時に損失値を表示するなど、学習の進捗状況をモニタリングできるようにせよ。

In [27]:
import torch
import torch.nn as nn
import torch.optim as optim

# モデル作成
model = BoWClassifier(E)

# 単語埋め込みは固定
model.embedding.weight.requires_grad = False

# 損失関数
criterion = nn.BCEWithLogitsLoss()

# optimizer: 学習するのは linear 層だけ
optimizer = optim.SGD(model.linear.parameters(), lr=0.1)

# 学習
num_epochs = 10

for epoch in range(num_epochs):

    model.train()
    total_loss = 0.0

    for example in train_dataset:

        input_ids = example["input_ids"]
        label = example["label"]

        # 勾配初期化
        optimizer.zero_grad()

        # 順伝播
        logits = model(input_ids)

        # 損失計算
        loss = criterion(logits, label)

        # 逆伝播
        loss.backward()

        # パラメータ更新
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_dataset)

    print(f"epoch: {epoch+1}, loss: {avg_loss:.4f}")

RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x6 and 300x1)

## 74. モデルの評価

問題73で学習したモデルの開発セットにおける正解率を求めよ。

In [11]:
def evaluate(model, dataset):
    model.eval()

    correct = 0

    with torch.no_grad():
        for example in dataset:
            logits = model(example["input_ids"])
            pred = (torch.sigmoid(logits) >= 0.5).float()
            correct += (pred == example["label"]).sum().item()

    return correct / len(dataset)

acc = evaluate(model, dev_dataset)
print(acc)

0.7259174311926605


## 75. パディング

複数の事例が与えられたとき、これらをまとめて一つのテンソル・オブジェクトで表現する関数`collate`を実装せよ。与えられた複数の事例のトークン列の長さが異なるときは、トークン列の長さが最も長いものに揃え、0番のトークンIDでパディングをせよ。さらに、トークン列の長さが長いものから順に、事例を並び替えよ。

例えば、訓練データセットの冒頭の4事例が次のように表されているとき、

```
[{'text': 'hide new secretions from the parental units',
  'label': tensor([0.]),
  'input_ids': tensor([  5785,     66, 113845,     18,     12,  15095,   1594])},
 {'text': 'contains no wit , only labored gags',
  'label': tensor([0.]),
  'input_ids': tensor([ 3475,    87, 15888,    90, 27695, 42637])},
 {'text': 'that loves its characters and communicates something rather beautiful about human nature',
  'label': tensor([1.]),
  'input_ids': tensor([    4,  5053,    45,  3305, 31647,   348,   904,  2815,    47,  1276,  1964])},
 {'text': 'remains utterly satisfied to remain the same throughout',
  'label': tensor([0.]),
  'input_ids': tensor([  987, 14528,  4941,   873,    12,   208,   898])}]
```

`collate`関数を通した結果は以下のようになることが想定される。

```
{'input_ids': tensor([
    [     4,   5053,     45,   3305,  31647,    348,    904,   2815,     47,   1276,   1964],
    [  5785,     66, 113845,     18,     12,  15095,   1594,      0,      0,      0,      0],
    [   987,  14528,   4941,    873,     12,    208,    898,      0,      0,      0,      0],
    [  3475,     87,  15888,     90,  27695,  42637,      0,      0,      0,      0,      0]]),
 'label': tensor([
    [1.],
    [0.],
    [0.],
    [0.]])}
```


In [12]:
import torch
from torch.nn.utils.rnn import pad_sequence

def collate(batch):
    # 長さが長い順に並び替え
    batch = sorted(
        batch,
        key=lambda x: len(x["input_ids"]),
        reverse=True
    )

    # input_ids だけ取り出す
    input_ids = [example["input_ids"] for example in batch]

    # label だけ取り出す
    labels = [example["label"] for example in batch]

    # パディング
    input_ids = pad_sequence(
        input_ids,
        batch_first=True,
        padding_value=0
    )

    # label をまとめる
    labels = torch.stack(labels)

    return {
        "input_ids": input_ids,
        "label": labels
    }

In [27]:
# 先頭4件を取得
batch = train_dataset[:4]

# collate実行
result = collate(batch)

# 表示
print(result)

{'input_ids': tensor([[    4,  5053,    45,  3305, 31647,   348,   904,  2815,    47,  1276,
          1964],
        [  987, 14528,  4941,   873,    12,   208,   898,     0,     0,     0,
             0],
        [ 5785,    66,    18,    12, 15095,  1594,     0,     0,     0,     0,
             0],
        [ 3475,    87, 15888,    90, 27695, 42637,     0,     0,     0,     0,
             0]]), 'label': tensor([[1.],
        [0.],
        [0.],
        [0.]])}


## 76. ミニバッチ学習

問題75のパディングの処理を活用して、ミニバッチでモデルを学習せよ。また、学習したモデルの開発セットにおける正解率を求めよ。

In [16]:
class BoWClassifier(nn.Module):

    def __init__(self, embedding_matrix):
        super().__init__()

        vocab_size, emb_dim = embedding_matrix.shape

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=emb_dim,
            padding_idx=0
        )

        self.embedding.weight.data.copy_(
            torch.tensor(embedding_matrix)
        )

        self.linear = nn.Linear(emb_dim, 1)

    def forward(self, input_ids):
        # input_ids: [batch_size, seq_len]

        x = self.embedding(input_ids)
        # x: [batch_size, seq_len, emb_dim]

        # PADでない場所
        mask = (input_ids != 0).unsqueeze(-1)
        # mask: [batch_size, seq_len, 1]

        # PAD部分を0にする
        x = x * mask

        # 各文の実トークン数
        lengths = mask.sum(dim=1)
        # lengths: [batch_size, 1]

        # 平均ベクトル
        x = x.sum(dim=1) / lengths
        # x: [batch_size, emb_dim]

        logits = self.linear(x)
        # logits: [batch_size, 1]

        return logits

In [30]:
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.optim as optim

# DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate
)

# モデル
model = BoWClassifier(E)

# 単語埋め込みは固定
model.embedding.weight.requires_grad = False

# 損失関数
criterion = nn.BCEWithLogitsLoss()

# optimizer
optimizer = optim.SGD(model.linear.parameters(), lr=0.1)

num_epochs = 10

for epoch in range(num_epochs):

    model.train()
    total_loss = 0.0

    for batch in train_loader:
        input_ids = batch["input_ids"]   # [batch_size, max_len]
        labels = batch["label"]          # [batch_size, 1]

        optimizer.zero_grad()

        logits = model(input_ids)        # [batch_size, 1]

        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(f"epoch: {epoch+1}, loss: {avg_loss:.4f}")

epoch: 1, loss: 0.5280
epoch: 2, loss: 0.4535
epoch: 3, loss: 0.4350
epoch: 4, loss: 0.4261
epoch: 5, loss: 0.4206
epoch: 6, loss: 0.4169
epoch: 7, loss: 0.4144
epoch: 8, loss: 0.4123
epoch: 9, loss: 0.4109
epoch: 10, loss: 0.4097


In [39]:
def accuracy(model, data_loader):
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"]
            labels = batch["label"]

            logits = model(input_ids)

            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).float()

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total

In [40]:
dev_acc = accuracy(model, dev_loader)

print(f"dev accuracy: {dev_acc:.4f}")

dev accuracy: 0.7683


## 77. GPU上での学習

問題76のモデル学習をGPU上で実行せよ。また、学習したモデルの開発セットにおける正解率を求めよ。

In [13]:
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [14]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate
)

In [17]:
model = BoWClassifier(E).to(device)

model.embedding.weight.requires_grad = False

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.SGD(model.linear.parameters(), lr=0.1)

num_epochs = 10

for epoch in range(num_epochs):

    model.train()
    total_loss = 0.0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        logits = model(input_ids)

        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(f"epoch: {epoch+1}, loss: {avg_loss:.4f}")

epoch: 1, loss: 0.5274
epoch: 2, loss: 0.4534
epoch: 3, loss: 0.4350
epoch: 4, loss: 0.4260
epoch: 5, loss: 0.4205
epoch: 6, loss: 0.4169
epoch: 7, loss: 0.4143
epoch: 8, loss: 0.4123
epoch: 9, loss: 0.4108
epoch: 10, loss: 0.4096


In [19]:
dev_acc = accuracy(model, dev_loader, device)
print(f"dev accuracy: {dev_acc:.4f}")

dev accuracy: 0.7844


## 78. 単語埋め込みのファインチューニング

問題77の学習において、単語埋め込みのパラメータも同時に更新するファインチューニングを導入せよ。また、学習したモデルの開発セットにおける正解率を求めよ。

In [20]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate
)

model = BoWClassifier(E).to(device)

# 単語埋め込みも更新する
model.embedding.weight.requires_grad = True

criterion = nn.BCEWithLogitsLoss()

# linear層 + embedding層 の両方を学習
optimizer = optim.SGD(model.parameters(), lr=0.1)

num_epochs = 10

for epoch in range(num_epochs):

    model.train()
    total_loss = 0.0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        logits = model(input_ids)

        loss = criterion(logits, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(f"epoch: {epoch+1}, loss: {avg_loss:.4f}")

cuda
epoch: 1, loss: 0.5158
epoch: 2, loss: 0.4167
epoch: 3, loss: 0.3813
epoch: 4, loss: 0.3590
epoch: 5, loss: 0.3422
epoch: 6, loss: 0.3289
epoch: 7, loss: 0.3178
epoch: 8, loss: 0.3085
epoch: 9, loss: 0.3007
epoch: 10, loss: 0.2938


In [21]:
dev_acc = accuracy(model, dev_loader, device)
print(f"dev accuracy: {dev_acc:.4f}")

dev accuracy: 0.8062


## 79. アーキテクチャの変更

ニューラルネットワークのアーキテクチャを自由に変更し、モデルを学習せよ。また、学習したモデルの開発セットにおける正解率を求めよ。例えば、テキストの特徴ベクトル（単語埋め込みの平均ベクトル）に対して多層のニューラルネットワークを通したり、畳み込みニューラルネットワーク（CNN; Convolutional Neural Network）や再帰型ニューラルネットワーク（RNN; Recurrent Neural Network）などのモデルの学習に挑戦するとよい。

In [23]:
import torch
import torch.nn as nn

class BoWMLPClassifier(nn.Module):

    def __init__(self, embedding_matrix, hidden_dim=128, dropout=0.3):
        super().__init__()

        vocab_size, emb_dim = embedding_matrix.shape

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=emb_dim,
            padding_idx=0
        )

        self.embedding.weight.data.copy_(
            torch.tensor(embedding_matrix, dtype=torch.float32)
        )

        self.mlp = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, input_ids):
        x = self.embedding(input_ids)

        mask = (input_ids != 0).unsqueeze(-1)
        x = x * mask

        lengths = mask.sum(dim=1)
        x = x.sum(dim=1) / lengths

        logits = self.mlp(x)

        return logits

In [24]:
from torch.utils.data import DataLoader
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate
)

model = BoWMLPClassifier(E, hidden_dim=128, dropout=0.3).to(device)

# ファインチューニングする場合
model.embedding.weight.requires_grad = True

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        logits = model(input_ids)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"epoch: {epoch+1}, loss: {avg_loss:.4f}")

cuda
epoch: 1, loss: 0.3483
epoch: 2, loss: 0.2467
epoch: 3, loss: 0.2086
epoch: 4, loss: 0.1854
epoch: 5, loss: 0.1686
epoch: 6, loss: 0.1559
epoch: 7, loss: 0.1445
epoch: 8, loss: 0.1354
epoch: 9, loss: 0.1264
epoch: 10, loss: 0.1204


In [25]:
dev_acc = accuracy(model, dev_loader, device)
print(f"dev accuracy: {dev_acc:.4f}")

dev accuracy: 0.7775
